# Euro area HICP: data, forecast and shock pass-through

The analysis behind the dashboard. The logic lives in `hicp/`; this notebook calls it and explains the results.

1. Data from the Eurostat API: cleaning, quality and consistency checks
2. Where inflation is: contributions (main aggregates and COICOP ver.2 divisions), countries, weights
3. 3-month forecast: base effects, models, out-of-sample evaluation, overfitting checks
4. Energy pass-through (local projections)
5. Shock simulator

Run `python -m hicp.pipeline` first; it downloads the data and writes `outputs/`. Set `DEMO = True` to run on synthetic data instead. Never show its numbers as euro area figures.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd

from hicp import plots
from hicp.config import load_config
from hicp.data import load_or_download
from hicp.synthetic import make_demo_data

# if charts don't display (e.g. some PyCharm versions), uncomment:
# import plotly.io as pio; pio.renderers.default = "notebook_connected"
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)

DEMO = os.getenv("HICP_DEMO", "0") == "1"   # or set DEMO = True
cfg = load_config()
data = make_demo_data(cfg) if DEMO else load_or_download(cfg)
print(f"Source: {data.source} | last month: {data.last_month} | {data.indices.shape[1]} series")

Source: Eurostat API | last month: 2026-09 | 484 series


## 1. Cleaning, quality and consistency checks
Cleaning rules:
- exact duplicates are dropped;
- conflicting duplicates stop the pipeline;
- non-positive or non-finite values are removed;
- nothing is interpolated.

Eurostat status flags are kept and reported. The consistency checks show the engine reproduces Eurostat's published numbers for both breakdowns.

In [2]:
print(data.cleaning)
if data.flags is not None:
    print(data.flags["flag"].value_counts().to_dict())
data.quality.head() if data.quality is not None else None

{'index exact duplicates dropped': 0, 'index values removed (non-positive or non-finite)': 0, 'item-weight exact duplicates dropped': 0, 'country-weight exact duplicates dropped': 0, 'observations with a status flag': 46850}
{'d': 41417, 'e': 5220, 'b': 176, 'u': 37}


,geo,coicop,first,last,observations,interior gaps,values removed,flags,latest flagged month
0,EA,CP01,1996-01,2026-09,369,0,0,{},NaN
1,EA,CP02,1996-01,2026-09,369,0,0,{},NaN
2,EA,CP03,1996-01,2026-09,369,0,0,{'u': np.int64(3)},2021-02 (u)
3,EA,CP04,1996-01,2026-09,369,0,0,{},NaN
4,EA,CP05,1996-01,2026-09,369,0,0,{'u': np.int64(3)},2021-02 (u)


In [3]:
from hicp.validation import run_checks
run_checks(data)

,group,check,value,tolerance,unit,passed,note
0,Weights,Country weights sum to 1000 (max |dev|),0.030,1.0,per mille,True,
1,Main aggregates,Weights sum to 1000 in every country (max |dev|),0.020,1.0,per mille,True,
2,Main aggregates,EA weights = sum of country weights x item wei...,0.006,1.0,per mille,True,
3,Main aggregates,EA headline YoY rebuilt from EA series (max |e...,0.015,0.1,pp,True,mean |error| 0.004
4,Main aggregates,EA headline YoY rebuilt from 21 countries (max...,0.020,0.1,pp,True,mean |error| 0.005
5,Main aggregates,Exact contributions sum to published YoY (max ...,0.015,0.1,pp,True,
6,Main aggregates,Missing values since 2015 (excl. latest month),0.000,0.0,cells,True,
7,COICOP divisions,Weights sum to 1000 in every country (max |dev|),0.030,1.0,per mille,True,
8,COICOP divisions,EA weights = sum of country weights x item wei...,0.006,1.0,per mille,True,
9,COICOP divisions,EA headline YoY rebuilt from EA series (max |e...,0.015,0.1,pp,True,mean |error| 0.005


## 2. Where inflation is

HICP is chain-linked every year with a December overlap. Within year $y$, with $D$ = December of $y-1$:

$$\frac{I_t}{I_D} = \sum_i w_i^{y}\,\frac{I_{i,t}}{I_{i,D}}$$

So the headline YoY is **not** a weighted mean of component YoY rates. The exact contributions (Ribe decomposition) are

$$C_{i,t} = \frac{I_D}{I_{t-12}}\,w_i^{y}\Big(\frac{I_{i,t}}{I_{i,D}}-1\Big) + \frac{I_{D'}}{I_{t-12}}\,w_i^{y-1}\,\frac{I_{i,D}-I_{i,t-12}}{I_{i,D'}},\qquad D'=\text{Dec}(y-2).$$

In [4]:
from hicp.aggregation import contributions, effective_weights

core = data.series(data.area, data.codes["core"]) if "core" in data.codes else None
plots.headline_chart(data.headline(), core).show()
contrib = contributions(data.area_components(), data.area_component_weights())
plots.contributions_chart(contrib, data.headline(), data.labels).show()

# same decomposition with the 13 COICOP ver.2 divisions (7 largest shown, the rest grouped)
div = contributions(data.area_breakdown(data.divisions), data.area_weights(data.divisions))
largest = list(data.area_weights(data.divisions).iloc[-1].sort_values(ascending=False).index[:7])
plots.contributions_chart(plots.keep_and_other(div, largest), data.headline(),
                          {**data.labels, plots.OTHER: "Other divisions"}).show()
div.tail(3).round(2)

coicop,CP01,CP02,CP03,CP04,CP05,CP06,CP07,CP08,CP09,CP10,CP11,CP12,CP13
2026-07,0.12,0.1,-0.00,0.55,0.02,0.20,0.96,0.07,0.16,0.04,0.46,0.08,0.19
2026-08,0.12,0.1,0.02,0.63,0.04,0.21,1.19,0.07,0.12,0.04,0.43,0.08,0.20
2026-09,0.16,0.1,-0.00,0.77,0.03,0.21,1.49,0.07,0.17,0.03,0.49,0.08,0.19


In [5]:
plots.country_heatmap(data).show()
omega = effective_weights(data.item_weights, data.country_weights, data.countries, data.components)
plots.weights_chart(omega.iloc[-1], data.labels, int(omega.index[-1])).show()

## 3. Forecast: headline YoY three months ahead

With $\pi$ the 12-month log change and $m$ the monthly log change,

$$\pi_{T+3} = \pi_T + \sum_{k=1}^{3}\big(m_{T+k} - m_{T+k-12}\big).$$

The $m_{T+k-12}$ terms are already published (base effects), so the forecast needs only three monthly changes.

**Recommended model:** the equal-weight average of ETS on the headline, ETS on the 5 components, a seasonal ARX on the components and a seasonal median on the components. It was chosen for robustness across 2013–2026 (see the README). Benchmarks: no change (random walk on inflation) and seasonal naive.

In [6]:
from hicp.forecast import ForecastSetup, final_forecast, base_effect_decomposition, RECOMMENDED

f = cfg["forecast"]
setup = ForecastSetup(data.area_components(), data.area_component_weights(), data.headline(),
                      h=f["horizon"], window=f["window"], seasonal_years=f["seasonal_years"])
ff = final_forecast(setup)
ff["models"].T.round(2)

,2026-10,2026-11,2026-12
No change (random walk),3.77,3.77,3.77
Seasonal naive,4.16,4.30,4.27
ETS headline,4.05,4.27,4.40
ETS bottom-up,3.96,4.09,4.16
Seasonal ARX bottom-up,3.99,4.02,4.09
Seasonal median bottom-up,3.84,3.81,3.92
4-model average (recommended),3.96,4.05,4.14


In [7]:
decomp = base_effect_decomposition(ff["index"], ff["origin"], setup.h)
plots.base_effects_chart(decomp).show()

### Out-of-sample evaluation
Each forecast uses only data available at its origin. The pipeline has already run the backtest; this cell reads `outputs/backtest.csv`. To recompute it, run `backtest(setup.forecasters(), data.headline(), f["backtest_start"])`, which takes about 2 minutes.

In [8]:
from hicp.evaluation import accuracy_by_window, interval_bands, realtime_coverage, spa_test
from hicp.forecast import BENCHMARKS, NO_CHANGE, backtest

out = cfg.output_dir / ("demo" if DEMO else "")
bt = pd.read_csv(out / "backtest.csv") if (out / "backtest.csv").exists() else backtest(setup.forecasters(), data.headline(), f["backtest_start"])
for c in ("origin", "target"):
    bt[c] = pd.PeriodIndex(bt[c], freq="M")
acc = accuracy_by_window(bt, setup.h, cfg.windows)
plots.accuracy_chart(acc[acc.window == "full"].drop(columns="window"), RECOMMENDED, BENCHMARKS).show()
acc.pivot(index="model", columns="window", values="rel. RMSE").round(3)

window,calm 2013-19,full,surge 2020-23,test,validation
model,,,,,
4-model average (recommended),0.925,0.767,0.722,0.926,0.762
ETS bottom-up,0.904,0.797,0.763,0.968,0.787
ETS headline,1.019,0.776,0.691,1.090,0.760
No change (random walk),1.000,1.000,1.000,1.000,1.000
Seasonal ARX bottom-up,1.005,0.830,0.786,0.925,0.829
Seasonal median bottom-up,1.013,0.902,0.886,0.851,0.909
Seasonal naive,0.950,0.908,0.894,1.001,0.908


### Overfitting checks
- **Data snooping:** the Hansen SPA test asks whether *any* candidate beats the no-change forecast, allowing for having searched over all of them.
- **Intervals:** real-time coverage of the 80% interval, built only from errors observed before each forecast, with and without volatility scaling.

In [9]:
candidates = [m for m in bt.model.unique() if m not in BENCHMARKS]
print({w: round(spa_test(bt, setup.h, candidates, NO_CHANGE, a, b), 3) for w, (a, b) in cfg.windows.items()})
pd.DataFrame([realtime_coverage(bt, RECOMMENDED, 3, 0.8, s) for s in (False, True)])

{'full': 0.085, 'validation': 0.086, 'test': 0.129, 'calm 2013-19': 0.377, 'surge 2020-23': 0.046}


,model,h,level,scaled,coverage,n
0,4-model average (recommended),3,0.8,False,0.664,125
1,4-model average (recommended),3,0.8,True,0.760,125


In [10]:
plots.track_record_chart(bt, data.headline(), [RECOMMENDED, NO_CHANGE]).show()
plots.fan_chart(data.headline(), ff["yoy"], interval_bands(bt, RECOMMENDED)).show()

## 4. Energy pass-through (local projections)

For each component $j$ and horizon $h=0,\dots,24$: $\;100[\ln I_{j,t+h}-\ln I_{j,t-1}] = \alpha_h + \beta_h x_t + \text{controls} + u_{t+h}$, with $x_t$ the monthly change in the energy HICP and HAC standard errors.

**Sample.** Both the origin month $t$ and the outcome month $t+h$ lie in 1997–2020: at horizon $h$ the last origin is December 2020 minus $h$ months, so the 2021–23 surge never enters, not even through $t+h$. The table compares this with the full sample, where the surge raises the estimates.

**Status: exploratory.** $x_t$ is the observed energy price change, not an identified shock (an oil-supply shock instrument, LP-IV, would be the rigorous upgrade), and pass-through is assumed identical across countries. The simulator therefore shows **direct effects by default** and treats indirect effects as an optional extension, with a range from the 90% bands of the component coefficients.

In [11]:
from hicp.passthrough import energy_passthrough

p = cfg["passthrough"]
sample = tuple(p["sample"])
lp = energy_passthrough(data.area_components(), energy=cfg.energy, horizons=p["horizons"], lags=p["lags"], sample=sample)
lp_full = energy_passthrough(data.area_components(), energy=cfg.energy, horizons=p["horizons"], lags=p["lags"])
print("observations by horizon (1997-2020 sample):", lp[cfg.energy]["n"].loc[[0, 12, 24]].to_dict())
plots.passthrough_chart(lp, data.labels, data.components, energy=cfg.energy, sample=sample).show()
pd.concat({"1997-2020": lp["ratio"], "full sample": lp_full["ratio"]}, axis=1).loc[[6, 12, 24]].round(3)

observations by horizon (1997-2020 sample): {0: 287, 12: 275, 24: 263}


1997-2020                         full sample                        
        SERV FOOD_P FOOD_NP IGD_NNRG        SERV FOOD_P FOOD_NP IGD_NNRG
h                                                                       
6      0.015  0.040   0.030    0.005       0.022  0.079   0.075    0.044
12     0.009  0.055   0.109    0.005       0.040  0.157   0.176    0.087
24     0.036  0.140   0.216    0.021       0.112  0.380   0.363    0.184

## 5. Shock simulator

The shock deviates a price level from its baseline, $I^{shock}_{c,i,t} = I^{base}_{c,i,t}(1+s_{c,i,t})$, and the euro area is re-aggregated exactly. As a rule of thumb, a permanent $x\%$ rise on basket share $\omega$ adds about $\omega x$ pp for 12 months, then drops out.

**Baseline.** Over the 3 forecast months the baseline *is* the central forecast (`anchor=ff["yoy"]`): every series is rescaled by a common factor so the aggregate reproduces it, and a scenario without shocks returns exactly the forecast. Beyond that, each series continues with its usual seasonal pattern and recent momentum. This 12–24-month path is a conditional assumption, not a forecast; forecast accuracy is only tested at 1–3 months. Impacts (shocked − baseline) barely depend on it.

**Oil example, sized with external data.** The pump price is $(c+\text{excise}+\text{margin})(1+\text{VAT})$, so its elasticity to the crude cost $c$ is $c/(c+\text{excise}+\text{margin})$. Brent and EUR/USD come from FRED (the pipeline writes `outputs/market.json`). The fuels share of energy comes from Eurostat's weights (`FUEL` / `NRG`).

In [12]:
import json
from hicp.shocks import Shock, simulate, fuel_price_elasticity
from hicp.market import fuels_share_of_energy

market = json.loads((out / "market.json").read_text()) if (out / "market.json").exists() else {}
eps = fuel_price_elasticity(market.get("brent_usd", 80), market.get("usd_per_eur", 1.10),
                            cfg["shocks"]["excise_eur_per_litre"], cfg["shocks"]["margin_eur_per_litre"])
share, source = fuels_share_of_energy(data, cfg["shocks"]["fuels_share_fallback"])
energy_move = 30 * eps * share
print(f"Brent {market.get('brent_usd')} $/bbl, EUR/USD {market.get('usd_per_eur')} -> elasticity {eps:.2f}; "
      f"fuels share {share:.3f} ({source}) -> energy HICP +{energy_move:.1f}%")
start = str(data.last_month + 1)
H = cfg["shocks"]["horizon"]
oil_shock = [Shock(cfg.energy, energy_move, start, shape="ramp", ramp_months=2)]

# null scenario = central forecast, exactly
null = simulate(data, [], horizon=H, anchor=ff["yoy"])
print("baseline - forecast (pp):", (null.yoy["baseline"].reindex(ff["yoy"].index) - ff["yoy"]).abs().max().round(12))

oil = simulate(data, oil_shock, horizon=H, energy=cfg.energy, anchor=ff["yoy"])  # direct effects (default)
plots.shock_chart(oil, data.labels, f"Brent +30% (energy HICP +{energy_move:.1f}%)").show()
oil.delta.iloc[12::3].round(3)

Brent 122.03 $/bbl, EUR/USD 1.1278 -> elasticity 0.44; fuels share 0.447 (Eurostat item weights 2026 (FUEL / NRG)) -> energy HICP +6.0%


baseline - forecast (pp): 0.0


,total,direct,indirect
2026-09,0.000,0.000,0.0
2026-12,0.660,0.660,0.0
2027-03,0.660,0.660,0.0
2027-06,0.661,0.661,0.0
2027-09,0.658,0.658,0.0
2027-12,0.000,0.000,0.0
2028-03,0.000,0.000,0.0
2028-06,0.000,0.000,0.0
2028-09,0.000,0.000,0.0


In [13]:
# exploratory: add indirect effects, with the range from the coefficients' 90% bands
from hicp.passthrough import ratio_bands

runs = {name: simulate(data, oil_shock, horizon=H, energy=cfg.energy, anchor=ff["yoy"], passthrough_ratio=r)
        for name, r in ratio_bands(lp, cfg.energy).items()}
pd.DataFrame({"direct only": oil.delta["total"], **{f"with indirect ({k})": v.delta["total"] for k, v in runs.items()}}
             ).iloc[12::3].round(3)

,direct only,with indirect (central),with indirect (low),with indirect (high)
2026-09,0.000,0.000,0.000,0.000
2026-12,0.660,0.668,0.586,0.750
2027-03,0.660,0.747,0.635,0.860
2027-06,0.661,0.808,0.643,0.974
2027-09,0.658,0.775,0.602,0.948
2027-12,0.000,0.141,0.028,0.253
2028-03,0.000,0.121,-0.003,0.244
2028-06,0.000,0.101,-0.010,0.212
2028-09,0.000,0.159,0.048,0.269


In [14]:
# several components and countries at once, at COICOP-division level (direct effects)
multi = simulate(data, [Shock("CP07", 10, start, countries=["FR"]), Shock("CP11", 5, start, countries=["FR", "ES"])],
                 horizon=H, codes=data.divisions, anchor=ff["yoy"])
plots.shock_chart(multi, data.labels, "Transport +10% in FR, restaurants & hotels +5% in FR and ES").show()
plots.shock_country_chart(multi).show()